# Official on Paper, Spoken on the Ground

Is the legally official language always the most widely spoken? This notebook
compares CLDR official-language status with per-country speaker-count rankings.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)

## 2 · Official language fields

Each `Country` exposes three CLDR-derived lists:
- `official_languages` — nationally official
- `official_regional_languages` — regionally official
- `de_facto_official_languages` — widely used without formal status

In [ ]:
ch = db.countries.get("CH")
print(f"{ch.label} ({ch.code})")
print(f"  Official:          {[l.label for l in ch.official_languages]}")
print(f"  Regional official: {[l.label for l in ch.official_regional_languages]}")
print(f"  De facto official: {[l.label for l in ch.de_facto_official_languages]}")

## 3 · Top spoken language per country (CLDR source)

In [ ]:
def top_spoken_language(country):
    cldr = [sc for sc in country.speaker_counts if sc.source == "cldr" and sc.speaker_count]
    if not cldr:
        return None, 0
    top = max(cldr, key=lambda sc: sc.speaker_count)
    return top.language, top.speaker_count

rows = []
for country in db.countries:
    top_lang, top_count = top_spoken_language(country)
    official = {l.part3 for l in country.official_languages}
    de_facto = {l.part3 for l in country.de_facto_official_languages}
    regional = {l.part3 for l in country.official_regional_languages}
    top_part3 = top_lang.part3 if top_lang else None
    rows.append({
        "country": country.label,
        "code": country.code,
        "continent": country.continent.label,
        "official_count": len(official),
        "regional_count": len(regional),
        "de_facto_count": len(de_facto),
        "top_spoken": top_lang.label if top_lang else None,
        "top_spoken_count": top_count,
        "top_is_official": top_part3 in official if top_part3 else None,
        "top_is_de_facto": top_part3 in de_facto if top_part3 else None,
        "official_labels": ", ".join(l.label for l in country.official_languages),
    })

df = pd.DataFrame(rows)
df.head(10)

## 4 · Bar chart — official language count per country

In [ ]:
multi = df[df["official_count"] > 0].sort_values("official_count", ascending=False).head(20)

fig = px.bar(
    multi,
    x="official_count",
    y="country",
    orientation="h",
    color="continent",
    text="official_count",
    labels={"official_count": "Official languages", "country": ""},
    title="Countries with the Most Official Languages",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=550)
fig.update_traces(textposition="outside")
fig.show()

## 5 · Mismatch countries

Countries where the most-spoken language (CLDR) is not nationally official.

In [ ]:
mismatch = df[(df["top_is_official"] == False) & df["top_spoken"].notna()].sort_values("top_spoken_count", ascending=False)
print(f"Countries where top-spoken language is not official: {len(mismatch)}")
mismatch[["country", "top_spoken", "top_spoken_count", "official_labels"]].head(15)

## 6 · Case studies

In [ ]:
for code in ("CH", "IN", "ZA", "US"):
    c = db.countries.get(code)
    top_lang, top_count = top_spoken_language(c)
    print(f"\n{c.label}")
    print(f"  Official: {[l.label for l in c.official_languages]}")
    print(f"  Top spoken (CLDR): {top_lang.label if top_lang else '—'} ({top_count:,})")
    cldr_sorted = sorted(
        [sc for sc in c.speaker_counts if sc.source == "cldr"],
        key=lambda sc: sc.speaker_count,
        reverse=True,
    )[:5]
    for sc in cldr_sorted:
        tag = ""
        if sc.language in c.official_languages:
            tag = " [official]"
        elif sc.language in c.de_facto_official_languages:
            tag = " [de facto]"
        print(f"    {sc.language.label}: {sc.speaker_count:,}{tag}")

## 7 · Summary

In [ ]:
has_official = (df["official_count"] > 0).sum()
aligned = (df["top_is_official"] == True).sum()
print(f"Countries with ≥1 official language: {has_official}")
print(f"Countries where top-spoken is official: {aligned}")
print(f"Countries with de facto official languages: {(df['de_facto_count'] > 0).sum()}")